In [ ]:
def plot_acquisition_heatmap(
    X,                   # shape (n_samples, 2): queried points
    acquisition_map,     # shape (grid_size**2,): acquisition values
    X_grid,              # shape (grid_size**2, 2): grid points
    next_query=None,     # shape (2,), optional: next query point
    title='Acquisition Heatmap',
    cmap='viridis',
    color_range=(-3, 1), # default color scale for consistency
    save_path=None       # optional: path to save plot
):
    grid_size = int(np.sqrt(len(acquisition_map)))
    acquisition_2d = acquisition_map.reshape(grid_size, grid_size)

    # Compute extent from grid
    x_min, x_max = X_grid[:, 0].min(), X_grid[:, 0].max()
    y_min, y_max = X_grid[:, 1].min(), X_grid[:, 1].max()
    extent = [x_min, x_max, y_min, y_max]

    plt.figure(figsize=(8, 6))
    im = plt.imshow(
        acquisition_2d,
        origin='lower',
        extent=extent,
        cmap=cmap,
        aspect='auto',
        vmin=color_range[0],
        vmax=color_range[1]
    )
    plt.colorbar(im, label='Acquisition Value')

    # Queried points
    plt.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', label='Queried Points')

    # Next query
    if next_query is not None:
        plt.plot(next_query[0], next_query[1], 'r*', markersize=14, label='Next Query')

    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()

In [ ]:
# Example DataFrame
def add_points_to_df(df, results):
    """
    Appends rows to df using:
    - x values from each result
    - an empty string for 'yield'
    - the method name as 'source'
    """
    for result in results:
        point = list(result['x'])
        row = point + [np.nan] + ['week-x']
        df.loc[len(df)] = row

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

def plot_output_points(df, scale_factor=1.0, yield_scale_factor=1.0, nan_marker='o'):
    """
    Plots the second-to-last column as 'Yield' and all other columns (except the last)
    as scaled variables, each with a unique color and no connecting lines.

    Parameters:
    - df: pandas DataFrame
    - scale_factor: numeric value to multiply non-yield variables (default is 1000)
    - nan_marker: marker style for points where 'yield' is missing (default is 'o')
    """
    columns = df.columns
    yield_col = columns[-2]
    other_cols = [col for col in df.columns if col != yield_col and col != columns[-1]]

    # Clean and convert yield column
    df[yield_col] = df[yield_col].replace('', np.nan)

    # Generate distinct colors
    cmap = cm.get_cmap('tab10', len(other_cols))

    # Plot yield values (non-NaN)
    if df[yield_col].notna().any():
        plt.plot(df[yield_col].astype(float) * yield_scale_factor, marker='x', linestyle='none', label=yield_col, color='black')

    # Plot missing yield points using nan_marker
    missing_yield_indices = df[yield_col].isna()
    if missing_yield_indices.any():
        plt.plot(df.index[missing_yield_indices], [0] * missing_yield_indices.sum(),
                 marker=nan_marker, linestyle='none', label='missing yield', color='gray')

    # Plot other variables scaled by scale_factor
    for i, col in enumerate(other_cols):
        plt.plot(df[col].astype(float) * scale_factor, marker='.', linestyle='none', label=col, color=cmap(i))

    plt.xlabel('Index')
    plt.ylabel('Output y')
    plt.title('Output Curve')
    plt.grid(True)
    plt.legend()
    plt.show()